# AutoInt+ 모델

- AutoInt 모델의 확장 버전
- 기존 AutoInt의 멀티헤드 셀프 어텐션(Multi-Head Self-Attention) 구조에 **병렬 DNN(Deep Neural Network)** 을 추가
- 어텐션 출력과 DNN 출력을 결합(concat)하여 최종 예측
- 명시적(explicit) 피처 상호작용 (어텐션) + 암묵적(implicit) 피처 상호작용 (DNN) 동시 학습

In [27]:
# 모듈 임포트
import time
import random
import pandas as pd
import numpy as np

from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, MaxPooling2D, Conv2D, Dropout, Lambda, Dense, Flatten, Activation, Input, Embedding, BatchNormalization, Concatenate
from tensorflow.keras.initializers import glorot_normal, Zeros, TruncatedNormal
from tensorflow.keras.regularizers import l2


from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.metrics import BinaryAccuracy


from tensorflow.keras.optimizers import Adam
from collections import defaultdict
import math

# Layer 정의

- 임베딩 레이어 : raw input 데이터를 저차원 임베딩 공간에 매핑

- 다층 퍼셉트론 : 비선형 레이어 (Dense Layer)를 쌓아 올린 구조

- 멀티 헤드 어텐션 : 쿼리(Query), 키(Key), 값(Value)에 따른 어텐션 계산 구조 (셀프 어텐션 구조)

> **AutoInt vs AutoInt+**: AutoInt+에서는 DNN이 어텐션과 **병렬(parallel)**로 동작하며, 두 출력을 결합하여 최종 예측을 수행합니다.

In [28]:
# 임베딩 레이어
class FeaturesEmbedding(Layer):
    '''
    임베딩 레이어에서 만약 피처 3개가 각각 x, y, z개의 고유값을 가질 때 피처 차원 = [x, y, z]
    전체 임베딩을 해야 할 개수 = x + y + z -> 총 (x+y+z) * 임베딩 차원 크기의 행렬 생성    
    '''

    def __init__(self, field_dims, embed_dim, **kwargs):
        super(FeaturesEmbedding, self).__init__(**kwargs)
        self.total_dim = sum(field_dims)
        self.embed_dim = embed_dim
        self.offsets = np.array((0, *np.cumsum(field_dims)[:-1]), dtype=np.int32)
        self.embedding = tf.keras.layers.Embedding(input_dim=self.total_dim, output_dim=self.embed_dim)

    def build(self, input_shape):
        # 임베딩 빌드 & 초기화
        self.embedding.build(input_shape)
        self.embedding.set_weights([tf.keras.initializers.GlorotUniform()(shape=self.embedding.weights[0].shape)])

    def call(self, x):
        # 들어온 입력의 임베딩을 가져오기
        x = x + tf.cast(tf.constant(self.offsets), x.dtype)
        return self.embedding(x)

# 다층 퍼셉트론
class MultiLayerPerceptron(Layer):  
    '''
    DNN 레이어입니다.
    - Tensorflow Keras에서는 Dense 레이어를 쌓아올린 구조
    - 필요에 따라 배치 정규화도 사용
    '''
    def __init__(self, input_dim, hidden_units, activation='relu', l2_reg=0, dropout_rate=0, use_bn=False, init_std=0.0001, output_layer=True):
        super(MultiLayerPerceptron, self).__init__()
        self.dropout_rate = dropout_rate
        self.use_bn = use_bn
        hidden_units = [input_dim] + list(hidden_units)
        if output_layer:
            hidden_units += [1]
        # Dense layer를 쌓아올립니다.
        self.linears = [Dense(units, activation=None, kernel_initializer=tf.random_normal_initializer(stddev=init_std),
                              kernel_regularizer=tf.keras.regularizers.l2(l2_reg)) for units in hidden_units[1:]]
        # 활성화 함수를 세팅합니다.
        self.activation = tf.keras.layers.Activation(activation)
        # 필요하다면 배치정규화도 진행합니다.
        if self.use_bn:
            self.bn = [BatchNormalization() for _ in hidden_units[1:]]
        self.dropout = Dropout(dropout_rate)

    def call(self, inputs, training=False):
        x = inputs
        for i in range(len(self.linears)):
            # input data가 들어오면 layer를 돌면서 벡터 값을 가져오게 됩니다.
            x = self.linears[i](x)
            if self.use_bn:
                x = self.bn[i](x, training=training)
            # 각 layer마다 나온 벡터 값에 활성화 함수와 dropout을 적용시켜 비선형성 구조와 과적합을 방지합니다.
            x = self.activation(x)
            x = self.dropout(x, training=training)
        return x

# 멀티 헤드 어텐션
class MultiHeadSelfAttention(Layer):  
    '''
    멀티 헤드 셀프 어텐션 레이어
    - 위에 작성한 수식과 같이 동작
    - 필요에 따라 잔차 연결(residual connection)도 진행
    '''
    def __init__(self, att_embedding_size=8, head_num=2, use_res=True, scaling=False, seed=1024, **kwargs):
        if head_num <= 0:
            raise ValueError('head_num must be a int > 0')
        self.att_embedding_size = att_embedding_size
        self.head_num = head_num
        self.use_res = use_res
        self.seed = seed
        self.scaling = scaling
        super(MultiHeadSelfAttention, self).__init__(**kwargs)

    def build(self, input_shape):
        if len(input_shape) != 3:
            raise ValueError(
                "Unexpected inputs dimensions %d, expect to be 3 dimensions" % (len(input_shape)))
        embedding_size = int(input_shape[-1])
        # 쿼리에 해당하는 매트릭스 
        self.W_Query = self.add_weight(name='query', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                       dtype=tf.float32,
                                       initializer=TruncatedNormal(seed=self.seed))
        # 키에 해당되는 매트릭스
        self.W_key = self.add_weight(name='key', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                     dtype=tf.float32,
                                     initializer=TruncatedNormal(seed=self.seed + 1))
        # 값(value)에 해당되는 매트릭스
        self.W_Value = self.add_weight(name='value', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                       dtype=tf.float32,
                                       initializer=TruncatedNormal(seed=self.seed + 2))
        # 필요하다면 잔차 연결 가능
        if self.use_res:
            self.W_Res = self.add_weight(name='res', shape=[embedding_size, self.att_embedding_size * self.head_num],
                                         dtype=tf.float32,
                                         initializer=TruncatedNormal(seed=self.seed))

        super(MultiHeadSelfAttention, self).build(input_shape)

    def call(self, inputs, **kwargs):
        if K.ndim(inputs) != 3:
            raise ValueError("Unexpected inputs dimensions %d, expect to be 3 dimensions" % (K.ndim(inputs)))
        
        # 입력이 들어오면 쿼리, 키, 값(value)에 매칭되어 각각의 값을 가지고 옴
        querys = tf.tensordot(inputs, self.W_Query, axes=(-1, 0))  
        keys = tf.tensordot(inputs, self.W_key, axes=(-1, 0))
        values = tf.tensordot(inputs, self.W_Value, axes=(-1, 0))

        # 헤드 개수에 따라 데이터를 분리
        querys = tf.stack(tf.split(querys, self.head_num, axis=2))
        keys = tf.stack(tf.split(keys, self.head_num, axis=2))
        values = tf.stack(tf.split(values, self.head_num, axis=2))
        
        # 쿼리와 키를 먼저 곱해줌. 위 이미지의 식 (5)와 같음
        inner_product = tf.matmul(querys, keys, transpose_b=True)
        if self.scaling:
            inner_product /= self.att_embedding_size ** 0.5
        self.normalized_att_scores =  tf.nn.softmax(inner_product)
        
        # 쿼리와 키에서 나온 어텐션 값을 값(value)에 곱해줌. 식 (6)과 같음
        result = tf.matmul(self.normalized_att_scores, values)
        # 식 (7)과 같이 쪼개어진 멀티 헤드를 모아주기
        result = tf.concat(tf.split(result, self.head_num, ), axis=-1)
        result = tf.squeeze(result, axis=0) 

        if self.use_res:
            result += tf.tensordot(inputs, self.W_Res, axes=(-1, 0))
        result = tf.nn.relu(result)
        
        # 결과 값을 리턴

        return result

    def compute_output_shape(self, input_shape):

        return (None, input_shape[1], self.att_embedding_size * self.head_num)

    def get_config(self, ):
        config = {'att_embedding_size': self.att_embedding_size, 'head_num': self.head_num, 'use_res': self.use_res,'seed': self.seed}
        base_config = super(MultiHeadSelfAttention, self).get_config()
        base_config.update(config)
        return base_config

In [29]:
# AutoInt+ 모델
# AutoInt 와의 차이점: 어텐션 출력과 병렬 DNN 출력을 결합(concat)하여 최종 예측 수행
class AutoIntPlus(Layer): 
    '''
    AutoInt+ 본체입니다.
    AutoInt에 병렬 DNN(Deep Neural Network)을 추가하여 
    명시적(어텐션) + 암묵적(DNN) 피처 상호작용을 동시에 학습합니다.
    
    구조:
    입력 → 임베딩 → ┬→ Multi-Head Self-Attention (명시적 상호작용)
                     └→ DNN (암묵적 상호작용)
                     ↓
                  Concat → Final Layer → Sigmoid
    '''
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2, att_res=True, 
                 l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False, dnn_dropout=0.4, init_std=0.0001,
                 dnn_hidden_units=(256, 128)):
        super(AutoIntPlus, self).__init__()
        # 임베딩 레이어를 정의합니다.
        self.embedding = FeaturesEmbedding(field_dims, embedding_size)
        self.num_fields = len(field_dims)
        self.embedding_size = embedding_size
        
        # [MY CODE] [어텐션 브랜치] 멀티 헤드 셀프 어텐션 레이어를 정의합니다.
        self.int_layers = [MultiHeadSelfAttention(att_embedding_size=embedding_size, head_num=att_head_num, use_res=att_res) for _ in range(att_layer_num)]
        
        # [MY CODE] [DNN 브랜치] 병렬로 동작하는 DNN 레이어를 정의합니다.
        self.dnn = MultiLayerPerceptron(
            input_dim=self.num_fields * embedding_size, 
            hidden_units=dnn_hidden_units, 
            activation='relu', 
            l2_reg=l2_reg_dnn, 
            dropout_rate=dnn_dropout, 
            use_bn=dnn_use_bn, 
            init_std=init_std,
            output_layer=False  # 마지막 출력 레이어는 합친 후에 적용
        )
        
        # 어텐션 출력 + DNN 출력을 결합한 후 최종 출력 레이어
        self.final_layer = Dense(1, use_bias=False, kernel_initializer=tf.random_normal_initializer(stddev=init_std))

    def call(self, inputs, training=False):
        # input 데이터에 해당되는 embedding 값을 가져옵니다.
        embed_output = self.embedding(inputs)
        
        # === 어텐션 브랜치 (명시적 상호작용) ===
        # 멀티 헤드 셀프 어텐션 레이어에서 상호작용을 수행합니다.
        att_input = embed_output
        for layer in self.int_layers:
            att_input = layer(att_input)
        att_output = Flatten()(att_input)
        
        # === DNN 브랜치 (암묵적 상호작용) ===
        # 임베딩을 Flatten 후 DNN에 통과시킵니다.
        dnn_input = Flatten()(embed_output)
        dnn_output = self.dnn(dnn_input, training=training)
        
        # === 결합 (Concat) ===
        # 어텐션 출력과 DNN 출력을 결합합니다.
        combined = tf.concat([att_output, dnn_output], axis=-1)
        
        # 최종 출력입니다.
        combined = self.final_layer(combined)
        # sigmoid로 예측값을 출력합니다.
        y_pred = tf.nn.sigmoid(combined)

        return y_pred

In [30]:
# 평가 함수는 아래의 링크에서 가져왔습니다.
# https://www.programcreek.com/python/?code=MaurizioFD%2FRecSys2019_DeepLearning_Evaluation%2FRecSys2019_DeepLearning_Evaluation-master%2FConferences%2FKDD%2FMCRec_our_interface%2FMCRecRecommenderWrapper.py
def get_DCG(ranklist, y_true):
    dcg = 0.0
    for i in range(len(ranklist)):
        item = ranklist[i]
        if item in y_true:
            dcg += 1.0 / math.log(i + 2)
    return  dcg

def get_IDCG(ranklist, y_true):
    idcg = 0.0
    i = 0
    for item in y_true:
        if item in ranklist:
            idcg += 1.0 / math.log(i + 2)
            i += 1
    return idcg

def get_NDCG(ranklist, y_true):
    '''NDCG 평가 지표'''
    ranklist = np.array(ranklist).astype(int)
    y_true = np.array(y_true).astype(int)
    dcg = get_DCG(ranklist, y_true)
    idcg = get_IDCG(y_true, y_true)
    if idcg == 0:
        return 0
    return round( (dcg / idcg), 5)

def get_hit_rate(ranklist, y_true):
    '''hitrate 평가 지표'''
    c = 0
    for y in y_true:
        if y in ranklist:
            c += 1
    return round( c / len(y_true), 5 )

In [31]:
# 모델 테스트
def test_model(model, test_df):
    '''모델 테스트'''
    user_pred_info = defaultdict(list)
    total_rows = len(test_df)
    for i in range(0, total_rows, batch_size):
        features = test_df.iloc[i:i + batch_size, :-1].values
        y_pred = model.predict(features, verbose=False)
        for feature, p in zip(features, y_pred):
            u_i = feature[:2]
            user_pred_info[int(u_i[0])].append((int(u_i[1]), float(p.item())))
    return user_pred_info

In [32]:
# 데이터 불러오고 세팅
# label encoder로 0부터 피처의 고유 개수까지 매핑 -> 훈련/테스트 데이터 분리

# 1. 데이터 불러오기
# csv 데이터이므로 read_csv로 가져옵니다.
movielens_rcmm = pd.read_csv("ml-1m/movielens_rcmm_v2.csv", dtype=str)
print(movielens_rcmm.shape)

# 2. 라벨 인코더(label encoder)
# label은 제외한 각 컬럼을 돌면서 각각의 고윳값들을 0부터 n까지 매핑시킵니다.
label_encoders = {col: LabelEncoder() for col in movielens_rcmm.columns[:-1]} # label은 제외

for col, le in label_encoders.items():
    movielens_rcmm[col] = le.fit_transform(movielens_rcmm[col])

movielens_rcmm['label'] = movielens_rcmm['label'].astype(np.float32)

# 3. 학습 데이터와 테스트데이터로 분리, 0.2 정도로 분리
train_df, test_df = train_test_split(movielens_rcmm, test_size=0.2, random_state=42)

# 필요 컬럼들과 레이블 정의
# 필드의 각 고유 개수를 정의하는 field_dims를 정의합니다. 이는  임베딩 때 활용됩니다. 
u_i_feature = ['user_id', 'movie_id']
meta_features = ['movie_decade', 'movie_year', 'rating_year', 'rating_month', 'rating_decade', 'genre1','genre2', 'genre3', 'gender', 'age', 'occupation', 'zip']
label = 'label'
field_dims = np.max(movielens_rcmm[u_i_feature + meta_features].astype(np.int64).values, axis=0) + 1
field_dims

(1000209, 15)


array([6040, 3706,   10,   81,    4,   12,    1,   18,   18,   16,    2,
          7,   21, 3439])

# 훈련 환경 및 모델 세팅

- 드롭아웃, 배치사이즈 등 모델 훈련, 설정 세팅
- AutoInt+ 모델 정의 (어텐션 + 병렬 DNN)

In [33]:
# 에포크, 학습률, 드롭아웃, 배치사이즈, 임베딩 크기 등 정의
epochs = 20          # 5 → 20 (Early Stopping이 자동으로 멈춤)
learning_rate = 0.001  # 0.0001 → 0.001 (초반 학습 속도 개선)
dropout = 0.3        # 0.4 → 0.3
batch_size = 2048
embed_dim = 32       # 16 → 32 (표현력 증가)

In [34]:
# AutoInt+ 레이어를 가지고 있는 모델 본체입니다. 해당 모델을 활용해 훈련을 진행합니다.
class AutoIntPlusModel(Model):
    def __init__(self, field_dims, embedding_size, att_layer_num=3, att_head_num=2
                 , att_res=True, l2_reg_dnn=0, l2_reg_embedding=1e-5, dnn_use_bn=False
                 , dnn_dropout=0, init_std=0.0001, dnn_hidden_units=(256, 128)):
        super(AutoIntPlusModel, self).__init__()
        self.autoIntPlus_layer = AutoIntPlus(field_dims, embedding_size, att_layer_num=att_layer_num, att_head_num=att_head_num, 
                                     att_res=att_res, l2_reg_dnn=l2_reg_dnn, dnn_dropout=dnn_dropout, init_std=init_std,
                                     dnn_hidden_units=dnn_hidden_units
                                    )

    def call(self, inputs, training=False):
        return self.autoIntPlus_layer(inputs, training=training)

# 모델 정의 (개선된 설정)
autoIntPlus_model = AutoIntPlusModel(
    field_dims, embed_dim, 
    att_layer_num=3, att_head_num=2, att_res=True,
    l2_reg_dnn=1e-5,               # 0 → 정규화 추가
    l2_reg_embedding=1e-5, 
    dnn_use_bn=True,               # False → True (배치 정규화 활성화)
    dnn_dropout=dropout, 
    init_std=0.0001, 
    dnn_hidden_units=(512, 256, 128)  # (256, 128) → 레이어 추가
)

# 옵티마이저, 오차함수 정의
optimizer = Adam(learning_rate=learning_rate)
loss_fn = BinaryCrossentropy(from_logits=False)

autoIntPlus_model.compile(optimizer=optimizer, loss=loss_fn, metrics=['binary_crossentropy'])

# 훈련 및 평가

In [35]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Early Stopping: val_loss가 3 에포크 연속 개선되지 않으면 자동 중단, 최적 가중치 복원
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
# Learning Rate Scheduler: val_loss가 2 에포크 개선되지 않으면 학습률을 절반으로 줄임
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6)

history = autoIntPlus_model.fit(
    train_df[u_i_feature + meta_features], train_df[label], 
    epochs=epochs, 
    batch_size=batch_size, 
    validation_split=0.1,
    callbacks=[early_stop, lr_scheduler]
)

Epoch 1/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 55s 145ms/step - binary_crossentropy: 0.5784 - loss: 0.5784 - val_binary_crossentropy: 0.5419 - val_loss: 0.5419 - learning_rate: 0.0010
Epoch 2/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 48s 136ms/step - binary_crossentropy: 0.5358 - loss: 0.5358 - val_binary_crossentropy: 0.5340 - val_loss: 0.5340 - learning_rate: 0.0010
Epoch 3/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 47s 132ms/step - binary_crossentropy: 0.5275 - loss: 0.5275 - val_binary_crossentropy: 0.5308 - val_loss: 0.5308 - learning_rate: 0.0010
Epoch 4/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 46s 130ms/step - binary_crossentropy: 0.5219 - loss: 0.5219 - val_binary_crossentropy: 0.5299 - val_loss: 0.5299 - learning_rate: 0.0010
Epoch 5/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 48s 136ms/step - binary_crossentropy: 0.5184 - loss: 0.5184 - val_binary_crossentropy: 0.5275 - val_loss: 0.5275 - learning_rate: 0.0010
Epoch 6/20
352/352 ━━━━━━━━━━━━━━━━━━━━ 48s 135ms/step - binary_crossentropy: 0.5147 - loss: 0.5147 - val_binary_cr

In [36]:
# 사용자에게 예측된 정보를 저장하는 딕셔너리 
user_pred_info = {}
# top10개
top = 10
# 테스트 값을 가지고 옵니다. 
mymodel_user_pred_info = test_model(autoIntPlus_model, test_df)
# 사용자마다 돌면서 예측 데이터 중 가장 높은 top 10만 가져옵니다. 
for user, data_info in tqdm(mymodel_user_pred_info.items(), total=len(mymodel_user_pred_info), position=0, leave=True):
    ranklist = sorted(data_info, key=lambda s : s[1], reverse=True)[:top]
    ranklist = list(dict.fromkeys([r[0] for r in ranklist]))
    user_pred_info[str(user)] = ranklist
# 원본 테스트 데이터에서 label이 1인 사용자 별 영화 정보를 가져옵니다.
test_data = test_df[test_df['label']==1].groupby('user_id')['movie_id'].apply(list)

100%|██████████| 6038/6038 [00:00<00:00, 86664.27it/s]


In [37]:
mymodel_ndcg_result = {}
mymodel_hitrate_result = {}

# 모델 예측값과 원본 테스트 데이터를 비교해서 어느정도 성능이 나왔는지 NDCG와 Hitrate를 비교합니다.

# NDCG
for user, data_info in tqdm(test_data.items(), total=len(test_data), position=0, leave=True):
    mymodel_pred = user_pred_info.get(str(user))

    testset = list(set(np.array(data_info).astype(int)))
    mymodel_pred = mymodel_pred[:top]

    # NDCG 값 구하기
    user_ndcg = get_NDCG(mymodel_pred, testset)

    mymodel_ndcg_result[user] = user_ndcg

# Hitrate
for user, data_info in tqdm(test_data.items(), total=len(test_data), position=0, leave=True):
    mymodel_pred = user_pred_info.get(str(user))

    testset = list(set(np.array(data_info).astype(int)))
    mymodel_pred = mymodel_pred[:top]

    # hitrate 값 구하기
    user_hitrate = get_hit_rate(mymodel_pred, testset)

    # 사용자 hitrate 결과 저장
    mymodel_hitrate_result[user] = user_hitrate

100%|██████████| 5994/5994 [00:00<00:00, 67625.32it/s]


In [38]:
print(" mymodel ndcg : ", round(np.mean(list(mymodel_ndcg_result.values())), 5))
print(" mymodel hitrate : ", round(np.mean(list(mymodel_hitrate_result.values())), 5))

 mymodel ndcg :  0.67131
 mymodel hitrate :  0.63681


# 저장

In [39]:
# 모델 저장
np.save('save_repo/field_dims.npy', field_dims)

# 가중치 저장
autoIntPlus_model.save_weights('save_repo/autoIntPlus_model.weights.h5')

import joblib 

# 모델 객체를 pickled binary file 형태로 저장
joblib.dump(label_encoders, 'save_repo/label_encoders.pkl')

['save_repo/label_encoders.pkl']

# 기록용

1. 1차
- mymodel ndcg :  0.66168
- mymodel hitrate :  0.63032

2. 2차 -> 하이퍼파라미터 튜닝만
- mymodel ndcg :  0.66587
- mymodel hitrate :  0.63557

3. 3차 -> 콜백 추가, DNN 구조 수정
- mymodel ndcg :  0.67087
- mymodel hitrate :  0.63697

4. 4차 -> 어텐션 헤드, 레이어 개수 확대
- mymodel ndcg :  0.67172
- mymodel hitrate :  0.63668

5. 피처 엔지니어링 + Hard Negative Sampling 적용
- mymodel ndcg :  0.67131
- mymodel hitrate :  0.63681